# 13 (v4): v4 final-model tile comparison (2 x 4)

Same per-tile figure and scoring as the v3 notebook 13, for the three **v4 final models**, trained on the v4 metrics (475 bands, no duplicate bands), each on its multi-seed **consensus top 100** tree-cover bands:

- **Top row:** MODIS VCF label (C6 2020, aggregated to 2 km), then Percent Tree Cover from XGBoost_v4_top100, Pixel_transformer_v4_top100, Spatial_CNN_v4_top100.
- **Bottom row:** PACE-period RGB (BandReflMedian Band_1/4/3, v4 metrics), then label-vs-prediction scatter for each model, coloured by absolute error.

Models (see the README "v4 plan"): XGBoost and Pixel-Transformer from `final_2026_10_14-10_10_01`; Spatial-CNN = consensus top 100 at model seed 23 from `sweep_cnncheck_2026_10_14-11_33_34` (MJ's pick, best of six CNN runs on the validation chips). Maps from `run_multi_model_inference.py infer --model all-v4` (289 of 291 tiles; h11v01 and h24v01 refused, as in v3).

**Which pixels are scored.** A pixel is used only if the label is valid (no nodata, no water), every model has a finite prediction, and no model had more than `MAX_IMPUTED_FRACTION` of its input bands missing there (AJ Kerr's inference fills missing inputs with the tile median, which would otherwise be scored against the label). NA values are always excluded. All models in a table are scored on identical pixels.

**R² here is the squared Pearson correlation** (r²) between label and prediction (MJ's choice); RMSE, MAE and bias are reported alongside. The chip-level tables in the README use the coefficient of determination instead -- don't mix the two.

The last cells compare the v4 finals with the v3 finals (XGBoost_v3_top200, Pixel_transformer_v3_top50, Spatial_CNN_v3_top200), and then with the v2 chip models too, each time on exactly the same pixels. Labels come from the v4 stacks (identical to the v3 stacks' labels). Outputs go to `multi_model_inference/comparison_plots_v4/`; the v3 plots in `comparison_plots_v3/` are not touched.


In [ ]:
from pathlib import Path
import zlib

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

PACE_YEAR = 2025
PACE_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/output")
INFER_BASE = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/multi_model_inference")
STACK_ROOT = Path("/explore/nobackup/projects/ilab/data/MODIS/PACE_VCF/full_tile_inference_input_v4")  # labels + masks
METRICS_SUBDIR = "3-Metrics-v4"  # for the RGB panel
OUT_DIR = INFER_BASE / "comparison_plots_v4"
RAW_BASE = INFER_BASE / "_raw_chip_predictions"  # per-tile imputed_feature_fraction.tif from the inference tool

# Skip pixels where any model had more than this share of its input bands
# missing (filled with the tile median). 0.5 drops pixels with essentially no
# PACE data; set 0.0 to score only pixels with every input present.
MAX_IMPUTED_FRACTION = 0.5

# v3 final models and v2 chip models, for the like-for-like tables in the last cells
V2_MODELS = [
    ("xgboost_chipped", "XGBoost_100runs"),
    ("pixel_transformer", "Pixel_transformer"),
    ("spatial_cnn", "Spatial_CNN"),
]
V3_MODELS = [
    ("xgboost_v3_top200", "XGBoost_v3_top200"),
    ("pixel_transformer_v3_top50", "Pixel_transformer_v3_top50"),
    ("spatial_cnn_v3_top200", "Spatial_CNN_v3_top200"),
]
OUT_DIR.mkdir(parents=True, exist_ok=True)

# (model_key, label) in column order
MODELS = [
    ("xgboost_v4_top100", "XGBoost_v4_top100"),
    ("pixel_transformer_v4_top100", "Pixel_transformer_v4_top100"),
    ("spatial_cnn_v4_top100", "Spatial_CNN_v4_top100"),  # consensus list, model seed 23
]

# TILES = ["h09v05", "h12v04", "h12v09", "h20v06", "h31v11"]  # AJ Kerr's comparison set
TILES = [t.strip() for t in open("/explore/nobackup/people/mfrost2/projects/PACE/tiles_all_291.txt") if t.strip()]

LABEL_NODATA = -255.0
WATER_VALUE = 200.0
METRICS_NODATA = -10001
RGB_BANDS = ("BandReflMedian-Band_1", "BandReflMedian-Band_4", "BandReflMedian-Band_3")
RGB_RANGE = (0.0, 2500.0)
SCATTER_MAX_POINTS = 50_000
SCATTER_ERROR_MAX = 25.0
SEED = 0

# Set False when running many tiles: inline figures are embedded in the .ipynb
# and can bloat it to 100+ MB (see 5c). PNGs are always saved to OUT_DIR.
SHOW_INLINE = False

In [ ]:
def stack_dir(tile):
    """Labels and masks come from the v4 stacks (the ones the v4 models read)."""
    if (STACK_ROOT / tile / "vcf_labels_2km.tif").exists():
        return STACK_ROOT / tile
    raise FileNotFoundError(f"no v4 stack/labels for {tile}")


def read_label(tile):
    d = stack_dir(tile)
    with rasterio.open(d / "vcf_labels_2km.tif") as src:
        labels = src.read().astype("float32")
    valid = np.all(np.isfinite(labels), axis=0) & np.all(labels != LABEL_NODATA, axis=0)
    valid &= ~np.any(labels == WATER_VALUE, axis=0)
    mask_path = d / "label_valid_mask.tif"
    if mask_path.exists():
        with rasterio.open(mask_path) as src:
            valid &= src.read(1).astype(bool)
    return labels[0], valid  # band 1 = Percent_Tree_Cover


def read_prediction(model_key, label, tile):
    path = INFER_BASE / model_key / f"{tile}_pcttree_{PACE_YEAR}_{label}.tif"
    if not path.exists():
        return None, None
    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")
        nodata = src.nodata
    valid = np.isfinite(data)
    if nodata is not None:
        valid &= data != nodata  # -255 for the chip models
    return data, valid


def read_imputed_ok(model_key, tile):
    """True where this model had at most MAX_IMPUTED_FRACTION of its inputs filled in."""
    path = RAW_BASE / model_key / tile / "imputed_feature_fraction.tif"
    if not path.exists():
        return None
    with rasterio.open(path) as src:
        frac = src.read(1).astype("float32")
    return np.isfinite(frac) & (frac <= MAX_IMPUTED_FRACTION)


def load_models(tile, models, valid):
    """Read each model's prediction and narrow `valid` to pixels usable for all of them.
    Returns (preds, valid), or (None, reason) if a model has no output for this tile."""
    preds = []
    for model_key, label in models:
        pred, pvalid = read_prediction(model_key, label, tile)
        if pred is None:
            return None, f"no {label} output"
        valid = valid & pvalid
        ok = read_imputed_ok(model_key, tile)
        if ok is not None:
            valid = valid & ok
        preds.append(pred)
    return preds, valid


def read_rgb(tile):
    path = PACE_BASE / tile / str(PACE_YEAR) / METRICS_SUBDIR / "MODIS_Metrics.tif"
    with rasterio.open(path) as src:
        idx = {d: i + 1 for i, d in enumerate(src.descriptions)}
        bands = np.stack([src.read(idx[b]).astype("float32") for b in RGB_BANDS])
    invalid = np.any(bands == METRICS_NODATA, axis=0)
    lo, hi = RGB_RANGE
    rgb = np.clip((np.moveaxis(bands, 0, -1) - lo) / (hi - lo), 0, 1)
    rgb[invalid] = 0.827  # light gray, as in AJ Kerr's plots
    return rgb


def stats(y, yhat):
    """R2 = squared Pearson correlation; NA (non-finite) values are excluded."""
    y = np.asarray(y, dtype="float64"); yhat = np.asarray(yhat, dtype="float64")
    ok = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[ok], yhat[ok]
    err = yhat - y
    r = np.corrcoef(y, yhat)[0, 1] if y.size > 1 and y.std() > 0 and yhat.std() > 0 else np.nan
    return {
        "r2": r ** 2,
        "pearson_r": r,
        "mae": np.mean(np.abs(err)),
        "rmse": np.sqrt(np.mean(err ** 2)),
        "bias": np.mean(err),
        "valid_pixels": int(y.size),
    }


def best_fit(x, y):
    x = x.astype("float64"); y = y.astype("float64")
    xc = x - x.mean()
    denom = np.sum(xc ** 2)
    if x.size < 2 or denom == 0:
        return None
    slope = np.sum(xc * (y - y.mean())) / denom
    return slope, y.mean() - slope * x.mean()

In [ ]:
def plot_tile(tile):
    target, valid = read_label(tile)
    preds, valid = load_models(tile, MODELS, valid)
    if preds is None:
        print(f"{tile}: {valid} -- skipping tile")
        return None
    if not valid.any():
        print(f"{tile}: no pixels with a valid label and real inputs for every model -- skipping")
        return None

    valid_idx = np.flatnonzero(valid.ravel())
    if valid_idx.size > SCATTER_MAX_POINTS:
        rng = np.random.default_rng(SEED + zlib.crc32(tile.encode()))
        scatter_idx = np.sort(rng.choice(valid_idx, SCATTER_MAX_POINTS, replace=False))
    else:
        scatter_idx = valid_idx

    cols = len(MODELS) + 1
    fig, axes = plt.subplots(2, cols, figsize=(4.1 * cols, 8.7), constrained_layout=True)
    map_cmap = plt.colormaps["Greens"].with_extremes(bad="lightgray")
    err_cmap = plt.colormaps["RdYlGn_r"]
    err_norm = Normalize(0, SCATTER_ERROR_MAX, clip=True)

    im = axes[0, 0].imshow(np.ma.array(target, mask=~valid), cmap=map_cmap, vmin=0, vmax=100,
                           interpolation="nearest")
    axes[0, 0].set_title("MODIS label (C6 2020)\nPercent_Tree_Cover")
    axes[1, 0].imshow(read_rgb(tile), interpolation="nearest")
    axes[1, 0].set_title("PACE input\nRGB (Band_1, Band_4, Band_3)")

    rows = []
    x_all = target.ravel()[scatter_idx]
    for col, ((model_key, label), pred) in enumerate(zip(MODELS, preds), start=1):
        axes[0, col].imshow(np.ma.array(pred, mask=~valid), cmap=map_cmap, vmin=0, vmax=100,
                            interpolation="nearest")
        axes[0, col].set_title(f"{label}\nprediction", fontsize=11)

        s = stats(target[valid], pred[valid])
        y = pred.ravel()[scatter_idx]
        ax = axes[1, col]
        ax.scatter(x_all, y, c=np.abs(y - x_all), cmap=err_cmap, norm=err_norm,
                   s=4, alpha=0.35, edgecolors="none", rasterized=True)
        ax.plot([0, 100], [0, 100], color="black", linewidth=1, label="1:1")
        fit = best_fit(x_all, y)
        if fit is not None:
            lx = np.array([0, 100])
            ax.plot(lx, fit[0] * lx + fit[1], color="black", linestyle="--", linewidth=1, label="fit")
        ax.set(title=f"r\u00b2={s['r2']:.3f}, bias={s['bias']:+.2f}\nMAE={s['mae']:.2f}, RMSE={s['rmse']:.2f}",
               xlabel="Label", ylabel="Prediction", xlim=(0, 100), ylim=(0, 100))
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7, loc="upper left")
        rows.append({"tile_id": tile, "model": label, **s})

    for ax in axes[0]:
        ax.axis("off")
    axes[1, 0].axis("off")
    fig.suptitle(f"v4 final models, full-tile comparison: {tile}\nPercent_Tree_Cover", fontsize=16)
    fig.colorbar(im, ax=axes[0, :], label="VCF percentage", shrink=0.78, pad=0.01)
    fig.colorbar(plt.cm.ScalarMappable(norm=err_norm, cmap=err_cmap), ax=axes[1, 1:],
                 label=f"Absolute error (percentage points; red \u2265 {SCATTER_ERROR_MAX:g})",
                 shrink=0.78, pad=0.01)

    out = OUT_DIR / f"{tile}_comparison_v4.png"
    fig.savefig(out, dpi=150)
    print(f"saved {out}")
    if SHOW_INLINE:
        plt.show()
    else:
        plt.close(fig)
    return rows

In [ ]:
all_rows, failed = [], []
for tile in TILES:
    try:
        rows = plot_tile(tile)
        if rows:
            all_rows.extend(rows)
    except Exception as e:
        failed.append((tile, repr(e)))
        print(f"{tile}: FAILED {e}")

metrics = pd.DataFrame(all_rows)
metrics.to_csv(OUT_DIR / "comparison_metrics_v4.csv", index=False)
print(f"{len(metrics) // len(MODELS)} tiles plotted, {len(failed)} failed")
metrics.round(3)


In [ ]:
# Pixel-weighted (pooled) summary across tiles -- not a plain mean of per-tile
# RMSEs, which over-weights small tiles (see unweighted-vs-pixel-weighted-rmse.md).
def pooled(g):
    n = g["valid_pixels"]
    return pd.Series({
        "tiles": len(g),
        "pooled_rmse": np.sqrt(np.sum(g["rmse"] ** 2 * n) / n.sum()),
        "pooled_mae": np.sum(g["mae"] * n) / n.sum(),
        "pooled_bias": np.sum(g["bias"] * n) / n.sum(),
        "median_tile_r2": g["r2"].median(),
        "mean_tile_r2": g["r2"].mean(),
    })

metrics.groupby("model").apply(pooled).round(3)

In [ ]:
# v3 final models vs v4 final models on EXACTLY the same pixels: valid label,
# finite prediction and real inputs (<= MAX_IMPUTED_FRACTION filled) for all six.
def same_pixels(models, csv_name):
    rows = []
    for tile in sorted(metrics.tile_id.unique()):
        try:
            target, valid = read_label(tile)
            preds, valid = load_models(tile, models, valid)
            if preds is None or not valid.any():
                continue  # a model has no map for this tile, or nothing left to score
            for (_, label), pred in zip(models, preds):
                rows.append({"tile_id": tile, "model": label, **stats(target[valid], pred[valid])})
        except Exception as e:
            print(f"{tile}: FAILED {e}")
    df = pd.DataFrame(rows)
    df.to_csv(OUT_DIR / csv_name, index=False)
    print(f"{df.tile_id.nunique()} tiles scored for all {len(models)} models (v4 alone: {metrics.tile_id.nunique()})")
    return df


v3v4 = same_pixels(V3_MODELS + MODELS, "comparison_metrics_v3_vs_v4_same_pixels.csv")
order = [label for pair in zip(V3_MODELS, MODELS) for _, label in pair]
v3v4.groupby("model")[["rmse", "mae", "bias", "r2", "valid_pixels"]].apply(pooled).round(3).loc[order]


In [ ]:
# v2 chip models, v3 finals and v4 finals together, on exactly the same pixels
# (fewer tiles: the v2 chip models cover fewer tiles).
v234 = same_pixels(V2_MODELS + V3_MODELS + MODELS, "comparison_metrics_v2_v3_v4_same_pixels.csv")
order = [label for trio in zip(V2_MODELS, V3_MODELS, MODELS) for _, label in trio]
v234.groupby("model")[["rmse", "mae", "bias", "r2", "valid_pixels"]].apply(pooled).round(3).loc[order]


In [ ]:
# The five standard comparison tiles (AJ Kerr's set), shown from the saved PNGs.
from IPython.display import Image, display

for tile in ["h09v05", "h12v04", "h12v09", "h20v06", "h31v11"]:
    png = OUT_DIR / f"{tile}_comparison_v4.png"
    if png.exists():
        display(Image(filename=str(png), width=1100))
    else:
        print(f"{tile}: no PNG yet ({png})")
